In [51]:
diretorio = r"G:\Drives compartilhados\Pedidos - TLV\Pedidos Pendentes\FARMA UTIL - 1 - 2024-07-26 16 55 00\test"
dfs = {}
contador = 0

In [52]:
import os

def listar_arquivos(directory_path):
    """
    Função para listar os arquivos em um diretório.

    :param directory_path: Caminho para o diretório a ser listado.
    :return: Lista de arquivos no diretório. Se o diretório não existir, retorna uma mensagem de erro.
    """
    try:
        # Verifica se o caminho é um diretório
        if not os.path.isdir(directory_path):
            return "O caminho fornecido não é um diretório ou não existe."
        
        # Lista os arquivos no diretório
        files = []
        for entry in os.listdir(directory_path):
            full_path = os.path.join(directory_path, entry)
            if os.path.isfile(full_path):
                files.append(entry)
        
        return files
    except Exception as e:
        return f"Ocorreu um erro: {str(e)}"


In [53]:

arquivos = listar_arquivos(diretorio)

In [54]:
arquivos

['374085.pdf']

In [64]:
import tabula
import re
import os
import shutil
import pandas as pd
from PyPDF2 import PdfReader
from openpyxl import Workbook, load_workbook



def processamento_arquivos(caminho_arquivo, logger=None):

        

    class ProcessamentoPDF:

        def __init__(self, caminho_arquivo):
            self.caminho_arquivo = caminho_arquivo
            self.lista_tabelas = tabula.read_pdf(caminho_arquivo, pages="all")
            self.reader = PdfReader(caminho_arquivo)
            self.text = self._extrair_texto_pdf()
            self.lista_retorno = []
            self.cnpjs, self.condicoes_pagamento, self.numero_pedido = self._extrair_informacoes_pdf()

        def _extrair_texto_pdf(self):
            text = ''
            for page in self.reader.pages:
                text += page.extract_text()
            return text

        def _extrair_informacoes_pdf(self):
            sections = re.split(r'(Código:)', self.text)
            cnpjs = []
            condicoes_pagamento = []
            numero_pedido = ""
            
            for i in range(1, len(sections), 2):
                section_text = sections[i] + sections[i+1]
                cnpj_filial_match = re.search(r'CNPJ:\s*(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})', section_text)
                condicao_pagamento_match = re.search(r'\n(.*?)Cotação:', section_text)
                numero_pedido_match = re.search(r'Cotação:\s*(\d+)', section_text)

                cnpj_filial = cnpj_filial_match.group(1) if cnpj_filial_match else ""
                condicao_pagamento = condicao_pagamento_match.group(1) if condicao_pagamento_match else ""
                if numero_pedido_match:
                    numero_pedido = numero_pedido_match.group(1)
                condicao_pagamento = condicao_pagamento.replace("DIAS", "").strip()

                cnpjs.append(cnpj_filial)
                condicoes_pagamento.append(condicao_pagamento)

            return cnpjs, condicoes_pagamento, numero_pedido

        def processar_tabelas(self):
            contador_index = 0
            contador = 0

            for index, tabela in enumerate(self.lista_tabelas):
        
                table = tabela.iloc[:, 0:]
                quantidade_linhas = len(table)
                
                lista_ean_extraido = []
                lista_desconto = []
                lista_quantidade = []
                produto_encontrado = False

                if "Unidade:" in table.columns:
                    contador_index -= 1

                for linha in range(quantidade_linhas):
                    
                    if all(col in table.columns for col in ["Produto", "Fabricante", "Preço Compra", "Desc. (%)", "Qtd.", "Total"]):
                        qtd = len(table.values)
                        cod_ean = table.iloc[linha, 0]
                        desconto_1 = table.iloc[linha, 4]
                        desconto_2 = table.iloc[linha, 5]
                        quantidade_1 = table.iloc[linha, 5]
                        quantidade_2 = table.iloc[linha, 6]
                        quantidade_3 = table.iloc[linha, 7] if qtd > 5 else None

                       

                        if not pd.isna(cod_ean):
                            lista_ean_extraido.append(int(cod_ean))
                        if not pd.isna(desconto_1) and "%" in str(desconto_1):
                            lista_desconto.append(desconto_1)
                        if not pd.isna(desconto_2) and "%" in str(desconto_2):
                            lista_desconto.append(desconto_2)
                        if not pd.isna(quantidade_1) and "," not in str(quantidade_1):
                            lista_quantidade.append(int(quantidade_1))
                        if not pd.isna(quantidade_2) and "," not in str(quantidade_2):
                            lista_quantidade.append(int(quantidade_2))
                        if not pd.isna(quantidade_3) and "," not in str(quantidade_3):
                            lista_quantidade.append(int(quantidade_3))

                        if lista_ean_extraido and lista_desconto and lista_quantidade:
                            pc_desconto = lista_desconto[0]
                            contador += 1
                            cnpj = self.cnpjs[contador_index]
                            cnpj = cnpj.replace("/","").replace("-","").replace(".", "").strip()
                            ean = lista_ean_extraido[0]
                            desconto = pc_desconto.replace("%", "").strip()
                            quantidade = lista_quantidade[0]
                          
                            dict_itens = {
                                "EAN": ean,
                                "O.C": self.numero_pedido,
                                "FILIAL": cnpj,
                                "DESCONTO": desconto,
                                "QUANTIDADE": quantidade
                            }

                            self.lista_retorno.append(dict_itens)
                            lista_ean_extraido = []
                            lista_desconto = []
                            lista_quantidade = []
                            produto_encontrado = True

                if produto_encontrado:
                    contador_index += 1

            return self.lista_retorno

        def exec(self):
            self._extrair_texto_pdf()
            self._extrair_informacoes_pdf()
            self.processar_tabelas()


    processador = ProcessamentoPDF(caminho_arquivo)
    df_limpo = processador.processar_tabelas()

    return df_limpo


In [65]:
for arquivo in arquivos:
    contador += 1
    df = processamento_arquivos(rf"{diretorio}\{arquivo}")
    print(df)
    dfs[contador] = df

0
1
2
3
4
5
6
7
8
[]


In [57]:
dfs

{1: []}

In [19]:


# records = []
# for key, values in dfs.items():
#     print(values)
#     for value in values:
#         print(value)
#         value['KEY'] = key  # Adicionar a chave principal ao registro
#         records.append(value)

# # Criar o DataFrame
# df = pd.DataFrame(records)

374092.pdf
374128.pdf
374104.pdf
374125.pdf
374117.pdf
374136.pdf
374100.pdf
374113.pdf
374108.pdf
374140.pdf
374085.pdf
374094.pdf


In [14]:
df

,EAN,O.C,FILIAL,DESCONTO,QUANTIDADE,KEY
0,7899674027672,12377,85035327000313,"30,69",2,6
1,7896015525583,12377,85035327000313,"50,00",1,6
2,7896015528577,12377,85035327000313,"50,00",2,6
3,7897595903914,12377,85035327000313,"39,65",1,6
4,11509031310,12377,85035327000313,"15,00",4,6
5,11509031334,12377,85035327000313,"15,00",3,6
6,7896049528505,12377,85035327000313,"5,00",1,6
7,7896342415694,12377,85035327000313,"15,00",1,6
8,7891653014215,12377,85035327000313,"15,00",1,6


In [36]:
df.to_excel(rf"C:\Users\Rafael\Downloads\farma-util\arquivo_farma-util1.xlsx")

In [13]:
df